# 🍅 Detector de Doenças em Folhas de Tomate

Projeto de portfólio: IA aplicada ao agro.

**Antes de começar:** no menu, vá em **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)** e salve.

Depois é só rodar as células **uma por uma, de cima para baixo** (Shift + Enter).

## 1. Importar as bibliotecas e conferir a GPU

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

print('TensorFlow:', tf.__version__)
print('GPU disponível:', tf.config.list_physical_devices('GPU'))

## 2. Baixar o dataset PlantVillage

São cerca de 54 mil fotos de folhas de várias plantas. O download leva alguns minutos.

In [ ]:
ds, info = tfds.load('plant_village', split='train', as_supervised=True, with_info=True)

todas_classes = info.features['label'].names
print('Total de classes no dataset:', len(todas_classes))

## 3. Ficar só com o tomate

Começar com uma cultura só deixa o projeto mais simples e o treino mais rápido.

In [ ]:
ids_tomate = [i for i, nome in enumerate(todas_classes) if nome.lower().startswith('tomato')]
classes = [todas_classes[i] for i in ids_tomate]

def nome_bonito(nome):
    return nome.replace('___', ' – ').replace('_', ' ')

print(f'{len(classes)} classes de tomate:')
for c in classes:
    print(' -', nome_bonito(c))

# Mapa: rótulo original -> novo rótulo (0, 1, 2...)
mapa = np.full(len(todas_classes), -1, dtype=np.int64)
for novo, antigo in enumerate(ids_tomate):
    mapa[antigo] = novo
mapa = tf.constant(mapa)
ids_tf = tf.constant(ids_tomate, dtype=tf.int64)

def e_tomate(idx, exemplo):
    _, rotulo = exemplo
    return tf.reduce_any(tf.equal(rotulo, ids_tf))

ds_tomate = ds.enumerate().filter(e_tomate)

## 4. Separar treino (80%) e validação (20%) e preparar as imagens

In [ ]:
TAM = 224
LOTE = 32

def preparar(idx, exemplo):
    imagem, rotulo = exemplo
    imagem = tf.image.resize(imagem, (TAM, TAM))
    return imagem, tf.gather(mapa, rotulo)

treino = (ds_tomate.filter(lambda i, ex: i % 5 != 0)
          .map(preparar, num_parallel_calls=tf.data.AUTOTUNE)
          .shuffle(1000).batch(LOTE).prefetch(tf.data.AUTOTUNE))

validacao = (ds_tomate.filter(lambda i, ex: i % 5 == 0)
             .map(preparar, num_parallel_calls=tf.data.AUTOTUNE)
             .batch(LOTE).prefetch(tf.data.AUTOTUNE))

# Mostrar algumas imagens de exemplo
plt.figure(figsize=(10, 10))
for imagens, rotulos in treino.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(imagens[i].numpy().astype('uint8'))
        plt.title(nome_bonito(classes[int(rotulos[i])]), fontsize=8)
        plt.axis('off')
plt.show()

## 5. Montar o modelo com Transfer Learning

Usamos a **MobileNetV2**, uma rede já treinada com milhões de imagens. Ela já sabe reconhecer formas, cores e texturas; só ensinamos a última camada a reconhecer as doenças. É rápida e leve o suficiente para rodar até no celular.

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=(TAM, TAM, 3), include_top=False, weights='imagenet')
base.trainable = False  # congela o que a rede já sabe

aumento = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

entrada = tf.keras.Input(shape=(TAM, TAM, 3))
x = aumento(entrada)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
saida = tf.keras.layers.Dense(len(classes), activation='softmax')(x)

modelo = tf.keras.Model(entrada, saida)
modelo.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo.summary()

## 6. Treinar

Com GPU, cada época leva alguns minutos. Fique de olho no `val_accuracy`: é a taxa de acerto em imagens que o modelo nunca viu.

In [ ]:
historico = modelo.fit(treino, validation_data=validacao, epochs=5)

## 7. Ver o resultado em gráfico

Esse gráfico é ótimo para colocar no README do GitHub.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(historico.history['accuracy'], label='Treino')
plt.plot(historico.history['val_accuracy'], label='Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.title('Acurácia do modelo')
plt.legend()
plt.grid(True)
plt.savefig('grafico_acuracia.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Acurácia final na validação: {historico.history['val_accuracy'][-1]:.1%}")

## 8. Salvar o modelo

Esses arquivos são usados no app com Streamlit.

In [ ]:
modelo.save('modelo_tomate.keras')
with open('classes.json', 'w') as f:
    json.dump([nome_bonito(c) for c in classes], f, ensure_ascii=False)
print('salvo')